# 12 — Dot Detector Parameter Tuning

Grid search over `min_similarity` and `max_gap_cap_hours` in `detect_dots`.
Evaluated on the 20 manually labeled stories using B-cubed P/R/F1.

**Current defaults:** `min_similarity=0.6`, `max_gap_cap_hours=24`

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from dot_detector import detect_dots

In [ ]:
df = pd.read_parquet(REPO_ROOT / "data/processed/articles_clean.parquet")
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

with open(REPO_ROOT / "data/processed/stories.pkl", "rb") as f:
    stories = pickle.load(f)

embs_all = np.load(REPO_ROOT / "data/processed/embeddings.npy")
emb_index = {i: embs_all[i] for i in range(len(df))}

# load ground truth
BUCKET_LABELS = {"2_5": "2-5", "5_10": "5-10", "10_20": "10-20", "20_50": "20-50", "50+": "50+"}
ground_truth = {}
for bucket in BUCKET_LABELS:
    with open(REPO_ROOT / f"data/evaluation/ground_truth_{bucket}.json") as f:
        for sid_str, v in json.load(f).items():
            ground_truth[int(sid_str)] = {**v, "bucket": BUCKET_LABELS[bucket]}

url_to_idx = {row["url"]: i for i, row in df.iterrows()}
print(f"GT stories: {len(ground_truth)}")

In [ ]:
def bcubed(gt_labels, sys_labels):
    gt = np.array(gt_labels)
    sys = np.array(sys_labels)
    precisions, recalls = [], []
    for i in range(len(gt)):
        same_sys = sys == sys[i]
        same_gt = gt == gt[i]
        precisions.append(np.sum(same_sys & same_gt) / np.sum(same_sys))
        recalls.append(np.sum(same_sys & same_gt) / np.sum(same_gt))
    p = float(np.mean(precisions))
    r = float(np.mean(recalls))
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


def evaluate_params(min_sim, max_gap):
    rows = []
    for sid, gt in ground_truth.items():
        arts = stories[sid]
        emb_idx = {i: emb_index[i] for i in arts}
        dots = detect_dots(arts, df, emb_idx, min_similarity=min_sim, max_gap_cap_hours=max_gap)

        idx_to_dot = {art_idx: d_id for d_id, dot in enumerate(dots) for art_idx in dot["indices"]}
        sys_labels = [idx_to_dot.get(url_to_idx.get(url), -1) for url in gt["articles"]]
        gt_labels = gt["labels"]

        p, r, f1 = bcubed(gt_labels, sys_labels)
        rows.append({"bucket": gt["bucket"], "P": p, "R": r, "F1": f1})

    res = pd.DataFrame(rows)
    overall = res[["P", "R", "F1"]].mean()
    by_bucket = res.groupby("bucket")[["F1"]].mean().round(3).reindex(["2-5", "5-10", "10-20", "20-50", "50+"])
    return overall["F1"], by_bucket["F1"].to_dict()

In [ ]:
MIN_SIMS = [0.55, 0.60, 0.65, 0.70]
MAX_GAPS = [6, 12, 18, 24, 36, 48]

grid_rows = []
for min_sim in MIN_SIMS:
    for max_gap in MAX_GAPS:
        overall_f1, by_bucket = evaluate_params(min_sim, max_gap)
        grid_rows.append({
            "min_similarity": min_sim,
            "max_gap_hours": max_gap,
            "F1_overall": round(overall_f1, 3),
            "F1_2-5": by_bucket.get("2-5", None),
            "F1_5-10": by_bucket.get("5-10", None),
            "F1_10-20": by_bucket.get("10-20", None),
            "F1_20-50": by_bucket.get("20-50", None),
            "F1_50+": by_bucket.get("50+", None),
        })
        print(f"sim={min_sim}  gap={max_gap:2}h  →  F1={overall_f1:.3f}  {by_bucket}")

grid = pd.DataFrame(grid_rows).sort_values("F1_overall", ascending=False)
print(f"\nBest overall:")
print(grid.head(5).to_string(index=False))

In [5]:
# pivot heatmap of overall F1
pivot = grid.pivot(index="min_similarity", columns="max_gap_hours", values="F1_overall")
pivot.index.name = "min_sim \\\ max_gap_h"
pivot.style.background_gradient(cmap="RdYlGn", axis=None).format("{:.3f}")

max_gap_hours,6,12,18,24,36,48
min_sim \\ max_gap_h,,,,,,
0.550000,0.852,0.862,0.858,0.847,0.825,0.813
0.600000,0.852,0.863,0.860,0.844,0.828,0.816
0.650000,0.857,0.867,0.866,0.857,0.839,0.822
0.700000,0.850,0.863,0.863,0.852,0.835,0.824


In [ ]:
# per-bucket F1 for the best overall config
best = grid.iloc[0]
print(f"Best config: min_similarity={best.min_similarity}  max_gap_hours={best.max_gap_hours}h")
print(f"  F1 overall : {best.F1_overall}")
print(f"  F1 2-5     : {best['F1_2-5']}")
print(f"  F1 5-10    : {best['F1_5-10']}")
print(f"  F1 10-20   : {best['F1_10-20']}")
print(f"  F1 20-50   : {best['F1_20-50']}")
print(f"  F1 50+     : {best['F1_50+']}")

In [7]:
import sys, pickle
import numpy as np
import pandas as pd
from pathlib import Path

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from dot_detector import detect_dots

df = pd.read_parquet(REPO_ROOT / "data/processed/articles_clean.parquet")
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

with open(REPO_ROOT / "data/processed/stories.pkl", "rb") as f:
    stories = pickle.load(f)

embs_all = np.load(REPO_ROOT / "data/processed/embeddings.npy")
emb_index = {i: embs_all[i] for i in range(len(df))}

from tqdm.notebook import tqdm

all_dots = {}
for sid, arts in tqdm(stories.items()):
    if arts:
        all_dots[sid] = detect_dots(arts, df, {i: emb_index[i] for i in arts})

with open(REPO_ROOT / "data/processed/dots_louvain_tuned.pkl", "wb") as f:
    pickle.dump(all_dots, f)

print("Done.")

  0%|          | 0/18473 [00:00<?, ?it/s]

Done.
